# Practical 08: Columnar Storage with Apache Parquet Format in PySpark

**UNIT 2: Big Data Processing using PySpark**

**Problem Statement:** Write a PySpark program to demonstrate high-performance columnar storage using Apache Parquet. Create a PySpark DataFrame, export it to Parquet format using .write.mode('overwrite').parquet(), reload the Parquet file using spark.read.parquet(), and verify schema preservation and row counts.

In [1]:
import os
import shutil
import tempfile
from pyspark.sql import SparkSession, Row

spark = (SparkSession.builder.appName("DS505-P08").master("local[*]")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

data = [
    {"Name": "Arjun", "Department": "IT", "City": "Pune", "Salary": 74000, "Rating": 4.4},
    {"Name": "Sneha", "Department": "HR", "City": "Delhi", "Salary": 56000, "Rating": 4.0},
    {"Name": "Vivek", "Department": "Finance", "City": "Pune", "Salary": 67000, "Rating": 3.8},
    {"Name": "Kriti", "Department": "IT", "City": "Jaipur", "Salary": 90000, "Rating": 4.7},
    {"Name": "Manav", "Department": "Sales", "City": "Delhi", "Salary": 49000, "Rating": 3.4},
]
emp_df = spark.createDataFrame([Row(**d) for d in data])
emp_df.printSchema()
emp_df.show()

tmp_dir = tempfile.mkdtemp()
path = os.path.join(tmp_dir, "staff_records.parquet")

emp_df.write.mode("overwrite").parquet(path)
print("Parquet written to:", path)
print("Files inside      :", [f for f in os.listdir(path) if not f.startswith(".")])

df_loaded = spark.read.parquet(path)
df_loaded.printSchema()
df_loaded.show()

print("Schema preserved :", emp_df.schema == df_loaded.schema)
print(f"Row count original = {emp_df.count()} | reloaded = {df_loaded.count()} | equal = {emp_df.count() == df_loaded.count()}")

df_loaded.select("Name").show()

shutil.rmtree(tmp_dir)
spark.stop()
print("Cleaned up.")

root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Rating: double (nullable = true)

+-----+----------+------+------+------+
| Name|Department|  City|Salary|Rating|
+-----+----------+------+------+------+
|Arjun|        IT|  Pune| 74000|   4.4|
|Sneha|        HR| Delhi| 56000|   4.0|
|Vivek|   Finance|  Pune| 67000|   3.8|
|Kriti|        IT|Jaipur| 90000|   4.7|
|Manav|     Sales| Delhi| 49000|   3.4|
+-----+----------+------+------+------+

Parquet written to: /tmp/tmp57nnsnqs/staff_records.parquet
Files inside      : ['part-00001-3087b444-7f92-4060-8c4f-1adaccc6072e-c000.snappy.parquet', '_SUCCESS', 'part-00000-3087b444-7f92-4060-8c4f-1adaccc6072e-c000.snappy.parquet']
root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Rating: double (nullable = true)

+-----+----------